# Exercise 4.3 — The decision threshold

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-03-thresholds.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

A logistic regression outputs a probability,

$$\hat{p}(x) = P(Y = 1 \mid x),$$

but a clinic needs a decision: test this person for diabetes, or not. To turn
the probability into a predicted class we choose a **decision threshold**.
Usually:

- if $\hat{p}(x) \geq 0.5$, predict 1 (diabetes);
- if $\hat{p}(x) < 0.5$, predict 0 (no diabetes).

But 0.5 is arbitrary. Changing the threshold changes how many true positives,
false positives, true negatives and false negatives the model makes, and so
which mistakes it makes.

**Theory:** decision threshold; confusion table; true and false positives and
negatives; sensitivity and specificity; the cost of each kind of mistake.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

The same 750 people and the same split as Exercises 4.1 and 4.2. We add a
column `diabetes_dummy`, the outcome coded 1 for `pos` and 0 for `neg`, which
makes the counting below easier to read.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["glucose", "mass", "age", "diabetes"]]
diabetes["diabetes_dummy"] = (diabetes["diabetes"] == "pos").astype(int) # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

------------------------------------------------------------------------

### Part 1 — From a probability to a predicted class

Fit the logistic regression from Exercise 4.2 on the training rows:

In [ ]:
log_fit = LogisticRegression(C=np.inf)
log_fit = log_fit.fit(train[["glucose"]], train["diabetes_dummy"])

print("intercept:", round(log_fit.intercept_[0], 4))
print("slope:    ", round(log_fit.coef_[0, 0], 4))

To make it easy to follow, predict the probability of diabetes for 15 people
from the test set only: the patients in rows 26 to 40. Python counts from 0, so
those are positions 25 to 39.

In [ ]:
patients15 = test.iloc[25:40].copy()

prediction_15 = patients15.copy()
prediction_15["prob"] = log_fit.predict_proba(patients15[["glucose"]])[:, 1]

prediction_15[["glucose", "diabetes_dummy", "prob"]].round(3)

Now turn each probability into a predicted class at three thresholds: 0.3, 0.5
and 0.7. A person is predicted to have diabetes (1) when their probability is
at or above the threshold:

In [ ]:
prediction_15["class_0.3"] = (prediction_15["prob"] >= 0.3).astype(int)
prediction_15["class_0.5"] = (prediction_15["prob"] >= 0.5).astype(int)
prediction_15["class_0.7"] = (prediction_15["prob"] >= 0.7).astype(int)

prediction_15[["glucose", "diabetes_dummy", "prob",
               "class_0.3", "class_0.5", "class_0.7"]].round(3)

Here are the 15 predicted probabilities, coloured by what is actually true, with
the three thresholds as dashed lines. Everyone above a line is predicted to have
diabetes at that threshold. The points follow the sigmoid curve from
Exercise 4.2.

In [ ]:
has_diabetes = prediction_15["diabetes_dummy"] == 1

fig, ax = plt.subplots()
ax.scatter(prediction_15.loc[has_diabetes, "glucose"], prediction_15.loc[has_diabetes, "prob"],
           color="darkorange", s=60, label="diabetes (1)")
ax.scatter(prediction_15.loc[~has_diabetes, "glucose"], prediction_15.loc[~has_diabetes, "prob"],
           color="grey", s=60, label="no diabetes (0)")
ax.axhline(0.3, color="blue",   linestyle="--")
ax.axhline(0.5, color="red",    linestyle="--")
ax.axhline(0.7, color="purple", linestyle="--")
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Predicted probability of diabetes")
ax.set_title("Predicted probabilities with the thresholds 0.3, 0.5 and 0.7")
ax.legend(title="What is true")
plt.show()

> **Learning points**
>
> - A classifier gives a probability; a threshold turns it into a predicted class.
> - The threshold is a choice, not part of the model. 0.5 is only a convention.
> - Lowering the threshold predicts diabetes for more people; raising it, for fewer.

------------------------------------------------------------------------

### Part 2 — Confusion tables, sensitivity and specificity

Each person ends up in one of four boxes, depending on what is true and what the
model predicted:

| | Predicted 1 (diabetes) | Predicted 0 (no diabetes) |
|---|:-:|:-:|
| **Truly 1** | True positive (**TP**) | False negative (**FN**): a case missed |
| **Truly 0** | False positive (**FP**): a false alarm | True negative (**TN**) |

This is the **confusion table**. **On paper first:** use the table of 15 people
above to count TP, FP, FN and TN at the threshold 0.5. Then check with code.

In [ ]:
pd.crosstab(prediction_15["diabetes_dummy"], prediction_15["class_0.5"],
            rownames=["True"], colnames=["Predicted"])

At 0.5 there are 4 true positives, 1 false positive, 3 false negatives and 7
true negatives. Now count all four boxes for the three thresholds side by side.
Two summaries of the table are used everywhere in medicine:

$$\text{Sensitivity} = \frac{TP}{TP + FN} \qquad\qquad
\text{Specificity} = \frac{TN}{TN + FP}$$

- **Sensitivity**: of the people who really have diabetes, the proportion the
  model catches.
- **Specificity**: of the people who really do not, the proportion the model
  correctly leaves alone.

Complete the code so that it counts the false negatives and computes the
specificity:

In [ ]:
# Replace every ______ below, then run this cell.
rows = []
for threshold in ["0.3", "0.5", "0.7"]:
    truth     = prediction_15["diabetes_dummy"]
    predicted = prediction_15["class_" + threshold]
    rows.append({"threshold": threshold,
                 "TP": ((truth == 1) & (predicted == 1)).sum(),
                 "FP": ((truth == 0) & (predicted == 1)).sum(),
                 "FN": (______).sum(),
                 "TN": ((truth == 0) & (predicted == 0)).sum()})

threshold_compare = pd.DataFrame(rows)
threshold_compare["sensitivity"] = threshold_compare["TP"] / (threshold_compare["TP"] + threshold_compare["FN"])
threshold_compare["specificity"] = ______

threshold_compare.round(3)

> **Stuck?**
>
> A false negative is a person who truly has diabetes (`truth == 1`) but was
> predicted not to (`predicted == 0`). Specificity uses the TN and FP columns, in
> the same way as sensitivity uses TP and FN.

Changing the threshold changes the kind of mistakes the model makes. A **low**
threshold catches more of the people with diabetes (higher sensitivity) at the
price of more false alarms. A **high** threshold raises fewer false alarms
(higher specificity) but misses more cases. At 0.7 this model misses six of the
seven people with diabetes.

> **Learning points**
>
> - The confusion table counts true positives, false positives, false negatives and true negatives.
> - Sensitivity = TP / (TP + FN); specificity = TN / (TN + FP).
> - Lowering the threshold raises sensitivity and lowers specificity, and the other way round.

------------------------------------------------------------------------

### Part 3 — The packaged version, on all 150 test patients

Fifteen people are enough to count by hand, not to judge a model. Whenever you
report a classification model you will build a confusion table, and packages
already do it for you. In scikit-learn it is `confusion_matrix()`:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

test_preds = test.copy()
test_preds["prob"] = log_fit.predict_proba(test[["glucose"]])[:, 1]
test_preds["predicted_0.5"] = (test_preds["prob"] >= 0.5).astype(int)

cm = confusion_matrix(test_preds["diabetes_dummy"], test_preds["predicted_0.5"])
cm

In [ ]:
display = ConfusionMatrixDisplay(cm, display_labels=["neg (0)", "pos (1)"])
display = display.plot(cmap="Blues")
plt.show()

> **Rows and columns are the other way round in R**
>
> `confusion_matrix()` puts the **truth** in the rows and the **prediction** in
> the columns. tidymodels' `conf_mat()` does the opposite. The numbers are the
> same; read the labels before you read the table.

At 0.5 the model finds 24 of the 53 people with diabetes (sensitivity 0.453)
and correctly leaves alone 90 of the 97 without (specificity 0.928). For a
screening test, missing more than half the cases is a poor result.

**Putting a price on the two mistakes.** Suppose a missed case (a false
negative) costs five times as much as an unnecessary test (a false positive):
the disease goes untreated, while the extra test costs one appointment. Then the
cost of a threshold is

$$\text{cost} = 5 \times FN + 1 \times FP$$

Work out the cost of every threshold from 0.1 to 0.9 on the test patients, and
find the cheapest:

In [ ]:
# Replace every ______ below, then run this cell.
rows = []
for threshold in np.round(np.arange(0.1, 1.0, 0.1), 1):
    truth = test_preds["diabetes_dummy"]
    prob  = test_preds["prob"]
    FN = ((truth == 1) & (prob <  threshold)).sum()
    FP = ((truth == 0) & (prob >= threshold)).sum()
    rows.append({"threshold": threshold, "FN": FN, "FP": FP, "cost": ______})

threshold_costs = pd.DataFrame(rows)
threshold_costs

**Question!** The cost ratio of five to one was an assumption. Who should decide
it, and what happens to the best threshold if a missed case costs twenty times
an unnecessary test instead of five?

> **Learning points**
>
> - Packages build the confusion table for you: `conf_mat()` in R, `confusion_matrix()` in Python. Check which way round the rows and columns are.
> - The best threshold depends on what each kind of mistake costs, which is a clinical decision, not a statistical one.
> - Report the threshold together with sensitivity and specificity: a threshold on its own says nothing.

## Stretch

> **Stretch — choose the threshold honestly**
>
> The cheapest threshold above was chosen by looking at the test patients, and
> then judged on the same test patients. Choose it on the **training** rows
> instead, then apply it once to the test patients. Is the cost still 71? Why is
> this the honest way round?